# qbank — Generador de preguntas de opción múltiple

`qbank` genera variantes de preguntas cuyas respuestas correctas se determinan automáticamente mediante lógica proposicional.

Este notebook recorre las principales funcionalidades. Ejecuta las celdas en orden con **Shift+Enter**.

**Contenidos:**
1. [Bloque básico: ProblemaTipo](#1)
2. [Revisión del banco: ProblemaTipoProfe](#2)
3. [Preguntas paramétricas con `setup`](#3)
4. [Preguntas de verdadero/falso: ProblemaVF](#4)
5. [Persistencia en JSON](#5)
6. [Banco de múltiples problemas](#6)
7. [Conversiones entre formatos](#7)
8. [Editor visual](#8)
9. [Preguntas multi-parte: ProblemaMultiParte](#9)
10. [Exportar a AMC (LaTeX)](#10)
11. [Exportar a Moodle (vía LaTeX)](#11)

In [ ]:
from qbank import *
import itertools

<a id="1"></a>
## 1. Bloque básico: `ProblemaTipo`

Un `ProblemaTipo` es una lista de componentes:
- **Cadenas de texto**: aparecen en todas las variantes.
- **Listas de `Supuesto`**: hipótesis alternativas (se elige una por variante).
- **Listas de `Cuestion`**: ítems de respuesta (se elige uno por variante).

Cada `Supuesto` y `Cuestion` tiene:
- `enunciado`: el texto que aparece en la pregunta.
- `semantica`: una fórmula lógica que determina si es verdadero o falso.

**Operadores disponibles** (variables con `v('nombre')`):

| Operador | Significado |
|----------|-------------|
| `v('A')` | variable proposicional A |
| `-v('A')` | negación de A |
| `v('A') & v('B')` | A y B |
| `v('A') \| v('B')` | A o B |
| `v('A') >> v('B')` | si A entonces B |
| `True` / `False` | siempre verdadero / siempre falso |

In [ ]:
p = ProblemaTipo([
    "Dado que ",
    [
        Supuesto("$\\mathcal{A}$ es verdadero, ", v('A')),
        Supuesto("$\\mathcal{B}$ es verdadero, ", v('B')),
    ],
    "indique qué afirmación es correcta: ",
    [
        Cuestion("$\\mathcal{A}$ es verdadero.", v('A')),
        Cuestion("$\\mathcal{B}$ es verdadero.", v('B')),
    ],
])

for etiqueta, enunciado, cuestiones in p:
    print(f"── Variante {etiqueta} ──")
    print(f"   {enunciado}")
    for texto, correcto, activa, exp in cuestiones:
        print(f"   {'✓' if correcto else '✗'} {texto}")
    print()

<a id="2"></a>
## 2. Revisión del banco: `ProblemaTipoProfe`

`ProblemaTipoProfe` funciona igual que `ProblemaTipo` pero **no descarta ninguna variante**:
muestra todas las combinaciones posibles, incluyendo las que serían inconsistentes o cuya
precondición no se cumple. Es la herramienta adecuada para revisar el banco de preguntas
antes de exportarlo y comprobar que todas las combinaciones tienen sentido pedagógico.

In [ ]:
for etiqueta, enunciado, cuestiones in ProblemaTipoProfe(p):
    print(f"── Variante {etiqueta} ──")
    print(f"   {enunciado}")
    for texto, correcto, activa, exp in cuestiones:
        marca = '✓' if correcto is True else ('✗' if correcto is False else '?')
        print(f"   {marca} {texto}")
    print()

<a id="3"></a>
## 3. Preguntas paramétricas con `setup`

El parámetro `setup` es una función sin argumentos que devuelve un diccionario de variables.
Se llama en cada variante, de modo que los valores numéricos o simbólicos cambian cada vez.
Los textos de los componentes admiten `@variable` para interpolación de esos valores.

Las semánticas pueden ser lambdas que reciben el diccionario `ns` generado por `setup`.

In [ ]:
import random

def numeros():
    a = random.randint(1, 9)
    b = random.randint(1, 9)
    return {'a': a, 'b': b, 'suma': a + b}

p_param = ProblemaTipo(
    [
        "Sean $a = @a$ y $b = @b$. ",
        [
            Cuestion("$a + b = @suma$", True),
            Cuestion("$a + b > 10$",   lambda ns: ns['a'] + ns['b'] > 10),
            Cuestion("$a = b$",        lambda ns: ns['a'] == ns['b']),
        ],
    ],
    setup=numeros,
)

for etiqueta, enunciado, cuestiones in itertools.islice(p_param, 4):
    print(f"── Variante {etiqueta} ──  {enunciado}")
    for texto, correcto, activa, exp in cuestiones:
        print(f"   {'✓' if correcto else '✗'} {texto}")
    print()

<a id="4"></a>
## 4. Preguntas de verdadero/falso: `ProblemaVF`

`ProblemaVF` genera variantes tomando aleatoriamente `NumPreguntas` ítems de un banco fijo
de cuestiones verdadero/falso. No usa lógica proposicional: la respuesta de cada ítem
es un booleano fijo.

In [ ]:
enunciado_vf = "Indique cuáles de las siguientes afirmaciones son verdaderas:"

banco_vf = [
    ("La derivada de $x^2$ es $2x$.",          True),
    ("La integral de $2x$ es $x^2 + C$.",       True),
    ("$\\sin^2(x) + \\cos^2(x) = 2$.",          False),
    ("El número $e$ es irracional.",             True),
    ("$\\ln(1) = 1$.",                           False),
    ("$\\sqrt{2}$ es irracional.",               True),
]

p_vf = ProblemaVF(enunciado_vf, banco_vf, NumPreguntas=3)

for etiqueta, enunciado, cuestiones in itertools.islice(p_vf, 3):
    print(f"── Variante {etiqueta} ──")
    for texto, correcto in cuestiones:
        print(f"   {'✓' if correcto else '✗'} {texto}")
    print()

<a id="5"></a>
## 5. Persistencia en JSON

`save_problema` y `load_problema` guardan y recuperan un problema (`ProblemaTipo`,
`ProblemaMultiParte` o `ProblemaVF`) en formato JSON sin perder ninguna información,
incluidos los `setup` que fueron definidos como cadenas de código.

In [ ]:
import os
os.makedirs("mis_problemas", exist_ok=True)

# Guardar
save_problema(p, "mis_problemas/ejemplo.json")
print("Guardado en mis_problemas/ejemplo.json")

# Cargar
p2 = load_problema("mis_problemas/ejemplo.json")
print("Cargado correctamente. Variantes:")
for etiqueta, enunciado, cuestiones in p2:
    print(f"  Variante {etiqueta}: {enunciado}")

<a id="6"></a>
## 6. Banco de múltiples problemas

`save_banco` y `load_banco` permiten agrupar varios problemas (de cualquier tipo) en un
único fichero JSON. Es el formato habitual cuando se trabaja con colecciones de ejercicios
para un curso o un examen.

In [ ]:
# Guardar varios problemas en un único fichero
save_banco([p, p_vf], "mis_problemas/banco.json")
print("Guardado banco con 2 problemas en mis_problemas/banco.json")

# Cargar el banco
problemas_cargados = load_banco("mis_problemas/banco.json")
print(f"Cargados {len(problemas_cargados)} problemas:")
for i, prob in enumerate(problemas_cargados):
    print(f"  Problema {i + 1}: {type(prob).__name__}")

<a id="7"></a>
## 7. Conversiones entre formatos

`qbank` usa el diccionario JSON como formato pivote. Desde cualquier representación
puedes obtener cualquier otra:

```
Lista de listas Python
      ↕  problema_to_dict() / problema_from_dict()
   Dict JSON  ←→  fichero .json  (save_problema / load_problema)
      ↓  problema_to_python()
  Código Python editable (.py)
```

In [ ]:
import json

# ── 1. ProblemaTipo → dict JSON ───────────────────────────────────────────────
d = problema_to_dict(p)
print("── Dict JSON (primeros 400 caracteres) ──")
print(json.dumps(d, ensure_ascii=False, indent=2)[:400], "...\n")

# ── 2. Dict JSON → ProblemaTipo ───────────────────────────────────────────────
p_desde_dict = problema_from_dict(d)
n_variantes = sum(1 for _ in p_desde_dict)
print(f"── ProblemaTipo recuperado del dict: {n_variantes} variantes ──\n")

# ── 3. ProblemaTipo → código Python editable ──────────────────────────────────
print("── Código Python generado por problema_to_python() ──")
print(problema_to_python(p))

In [ ]:
# Guardar el código Python en un fichero editable
save_problema_py(p, "mis_problemas/ejemplo.py")
print("Guardado en mis_problemas/ejemplo.py")
print("El fichero puede abrirse con cualquier editor y ejecutarse directamente con Python.")

<a id="8"></a>
## 8. Editor visual

`ProblemaTipoEditor` es un formulario interactivo para diseñar problemas sin escribir
la lista de listas a mano. Requiere `ipywidgets`.

| Botón | Acción |
|-------|--------|
| **▶ Preview** | muestra las primeras *n* variantes generadas |
| **💾 Guardar** | escribe el JSON en el fichero indicado |
| **📂 Cargar** | carga el JSON del fichero indicado |
| **{ } JSON** | muestra el JSON en el área de salida |
| **⬇ Descargar** | descarga el JSON directamente al navegador |

### 8.1 Editor vacío

In [ ]:
from qbank import ProblemaTipoEditor

editor = ProblemaTipoEditor()

> **Nota**: si el editor muestra texto en lugar del formulario (`VBox(children=...`),
> recarga la página y vuelve a ejecutar las celdas desde el principio.

### 8.2 Editor cargando un ejemplo con `setup`

El editor también puede cargar un problema existente. A continuación se carga
`desv_tipica.json`, un ejemplo con `setup` paramétrico que genera valores numéricos
distintos en cada variante. Puedes modificarlo en el formulario y volver a guardarlo
o descargarlo con **⬇ Descargar**.

In [ ]:
editor_setup = ProblemaTipoEditor('desv_tipica.json')

<a id="9"></a>
## 9. Preguntas multi-parte: `ProblemaMultiParte`

`ProblemaMultiParte` agrupa varias sub-preguntas bajo un enunciado común.
Cada `SubPregunta` tiene su propio texto introductorio y sus propias opciones;
todas se evalúan y se muestran marcadas como correctas o incorrectas.

Es el formato apropiado para las preguntas *multi-parte* de AMC y *cloze* de Moodle.

In [ ]:
p_multi = ProblemaMultiParte(
    componentes=[
        "Una desviación típica es un indicador ",
        [
            Supuesto("de dispersión. ",        v('Disp')),
            Supuesto("de tendencia central. ", -v('Disp')),
        ],
    ],
    subpreguntas=[
        SubPregunta("(en cuanto a su objetivo)",
            [Cuestion("de tendencia central", -v('Disp')),
             Cuestion("de dispersión",          v('Disp'))]),
        SubPregunta("(en cuanto a su sensibilidad)",
            [Cuestion("sensible a valores extremos",   v('Disp')),
             Cuestion("no muy sensible a extremos",   -v('Disp'))]),
    ]
)

for etiqueta, enunciado, subpreguntas in p_multi:
    print(f"── Variante {etiqueta} ──")
    print(f"   {enunciado}")
    for intro, cuestiones in subpreguntas:
        print(f"   {intro}")
        for texto, correcto, _, _ in cuestiones:
            print(f"     {'✓' if correcto else '✗'} {texto}")
    print()

<a id="10"></a>
## 10. Exportar a AMC (LaTeX)

Las funciones `AMC*` generan el código LaTeX para **Auto Multiple Choice**.
El fichero resultante se incluye en el documento AMC con `\input{fichero.tex}`.

### 10.1 Pregunta estándar (`AMC`)

In [ ]:
import os
os.makedirs("exportaciones", exist_ok=True)

with open("exportaciones/preguntas_amc.tex", "w") as f:
    for etiqueta, enunciado, cuestiones in p:
        f.write(AMC("MiCuestionario", etiqueta, enunciado, cuestiones))

with open("exportaciones/preguntas_amc.tex") as f:
    print(f.read())

### 10.2 Pregunta multi-parte (`AMC_multipart`)

Genera un único `\begin{questionmult}` con un bloque `\begin{choices}` por sub-pregunta.

In [ ]:
with open("exportaciones/preguntas_multi_amc.tex", "w") as f:
    for etiqueta, enunciado, subpreguntas in p_multi:
        f.write(AMC_multipart("Estadistica", etiqueta, enunciado, subpreguntas))

with open("exportaciones/preguntas_multi_amc.tex") as f:
    print(f.read())

<a id="11"></a>
## 11. Exportar a Moodle (vía LaTeX)

Las funciones `Quiz*` generan un fichero `.tex` que, al compilarlo con `xelatex`,
produce un `.xml` importable en Moodle.

```bash
xelatex MiCuestionario.tex   # genera MiCuestionario.xml
```

Importar en Moodle: **Banco de preguntas → Importar → Formato Moodle XML**.

### 11.1 Pregunta estándar (`QuizMoodleLastCh`)

In [ ]:
QuizMoodleLastCh("MiCuestionario", "exportaciones/", p)
print("Generado exportaciones/MiCuestionario.tex")

with open("exportaciones/MiCuestionario.tex") as f:
    print(f.read()[:800], "...")

### 11.2 Pregunta cloze multi-parte (`QuizClozeMulti`)

Genera preguntas `\begin{cloze}` con sub-bloques `\begin{multi}` incrustados,
una por cada `SubPregunta`.

In [ ]:
QuizClozeMulti("Estadistica", "exportaciones/", p_multi)
print("Generado exportaciones/Estadistica.tex")

with open("exportaciones/Estadistica.tex") as f:
    print(f.read()[:800], "...")

---
## Siguiente paso

Consulta el **Manual completo** en `Manual.org` para más detalles sobre:
- Bancos masivos con `setup` paramétrico (dos patrones de exportación)
- Opciones de exportación AMC (`AMCmc`, `AMClastCh`, `AMCmcProfe`, …)
- Cómo usar `ProblemaTipoEditor` para construir y exportar problemas sin escribir código